In [1]:
import pandas as pd
import numpy as np
import pyodbc
from datetime import datetime

In [2]:
query = """
SELECT
    gm.ACID,
    gm.FORACID,
    gm.ACCT_NAME,
    gm.CIF_ID,
    gm.SOL_ID,
    gm.ACCT_OPN_DATE,
    gm.SCHM_TYPE,
    gm.SCHM_CODE,
    gm.ACCT_CRNCY_CODE,

    hd.TRAN_DATE,
    hd.TRAN_ID,
    hd.TRAN_AMT,
    hd.TRAN_PARTICULAR,
    hd.TRAN_RMKS,
    hd.PART_TRAN_TYPE,
    hd.TRAN_TYPE,
    hd.TRAN_SUB_TYPE,
    hd.RCRE_TIME,
    hd.RPT_CODE

FROM RSVRRealtimeFinacleUGPRD.TBAADM.HIST_TRAN_DTL_TABLE hd

JOIN RSVRRealtimeFinacleUGPRD.TBAADM.GENERAL_ACCT_MAST_TABLE gm
ON hd.ACID = gm.ACID

WHERE
    gm.ACCT_CRNCY_CODE = 'USD'
    AND gm.ACCT_OPN_DATE >= '2024-01-01'
    AND gm.ACCT_CLS_FLG = 'N'
"""

In [3]:
conn = pyodbc.connect(
    'Driver={ODBC Driver 17 for SQL Server};'
    'Server=PUGSQLRSVRAG02.ug.sbicdirectory.com,5200;'
    'UID=RSVR_P_SQLACC_R_DS;'
    'Database=RSVRRealtimeFinacleUGPRD;'
    'PWD=M5tekitu45t65!g;'
    'Trusted_Connection=no;'
)

In [4]:
df = pd.read_sql(query, conn)

df.shape

/tmp/ipykernel_1907844/1634839272.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


(1929017, 19)

In [5]:
txns = df.copy()

In [6]:
fee_patterns = [
    'FEE',
    'STAMP DUTY',
    'EXCISE',
    'SMS',
    'COMMISSION',
    'CHARGE',
    'INTEREST'
]

In [13]:
df['TRAN_PARTICULAR'].fillna('').str.upper().str.contains('|'.join(fee_patterns))

0          False
1          False
2          False
3          False
4          False
           ...  
1929012    False
1929013    False
1929014    False
1929015    False
1929016    False
Name: TRAN_PARTICULAR, Length: 1929017, dtype: bool

In [14]:
mask = (
    ~txns['TRAN_PARTICULAR'].fillna('').str.upper().str.contains('|'.join(fee_patterns))
    &
    (txns['RPT_CODE'] != 'CHG')
)

txns = txns.loc[mask].copy()

In [15]:
txns[
    txns['ACCT_NAME']
    .str.contains('FRONTIER', case=False, na=False)
][
    [
        'ACCT_NAME',
        'FORACID',
        'ACCT_OPN_DATE',
        'SCHM_TYPE',
        'SOL_ID'
    ]
]

,ACCT_NAME,FORACID,ACCT_OPN_DATE,SCHM_TYPE,SOL_ID
343972,FRONTIER RESOURCES LIMITED,9030025780417,2025-01-16,ODA,031060
343973,FRONTIER RESOURCES LIMITED,9030025780417,2025-01-16,ODA,031060
346180,FRONTIER RESOURCES LIMITED,9030025780417,2025-01-16,ODA,031060
668231,FRONTIER RESOURCES LIMITED,9030025780417,2025-01-16,ODA,031060
767160,FRONTIER RESOURCES LIMITED,9030025780417,2025-01-16,ODA,031060
792170,FRONTIER RESOURCES LIMITED,9030026724367,2025-07-24,ODA,031032
884344,FRONTIER RESOURCES LIMITED,9030025780417,2025-01-16,ODA,031060
920801,FRONTIER RESOURCES LIMITED,9030025780417,2025-01-16,ODA,031060
969937,FRONTIER RESOURCES LIMITED,9030025780417,2025-01-16,ODA,031060
979844,FRONTIER RESOURCES LIMITED,9030025780417,2025-01-16,ODA,031060


In [16]:
import re

def clean_counterparty(text):
    if pd.isna(text):
        return None

    text = str(text).strip().upper()

    # Remove leading numbering like "1/"
    text = re.sub(r'^\d+/', '', text)

    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text)

    return text

txns['COUNTERPARTY'] = txns['TRAN_PARTICULAR'].apply(clean_counterparty)

In [18]:
txns['TRAN_PARTICULAR']

0          POS PURCHASE        aliexpress             408 78
1          POS PURCHASE        aliexpress             114 08
2          POS PURCHASE        aliexpress             408 78
3          POS PURCHASE        aliexpress             408 78
4          POS PURCHASE        aliexpress             408 78
                                 ...                        
1929012                          Payments Services - Savings
1929013                                     Wildmist booking
1929014                                     TECHTHINGS INTTX
1929015                                          DIANA ACAYE
1929016                             WOLDU KESETE HABTEYESUSE
Name: TRAN_PARTICULAR, Length: 1082247, dtype: object

In [17]:
txns['COUNTERPARTY']

0          POS PURCHASE ALIEXPRESS 408 78
1          POS PURCHASE ALIEXPRESS 114 08
2          POS PURCHASE ALIEXPRESS 408 78
3          POS PURCHASE ALIEXPRESS 408 78
4          POS PURCHASE ALIEXPRESS 408 78
                        ...              
1929012       PAYMENTS SERVICES - SAVINGS
1929013                  WILDMIST BOOKING
1929014                  TECHTHINGS INTTX
1929015                       DIANA ACAYE
1929016          WOLDU KESETE HABTEYESUSE
Name: COUNTERPARTY, Length: 1082247, dtype: object

In [19]:
txns['ACID'].nunique()

20512

In [20]:
txns.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1082247 entries, 0 to 1929016
Data columns (total 20 columns):
 #   Column           Non-Null Count    Dtype         
---  ------           --------------    -----         
 0   ACID             1082247 non-null  object        
 1   FORACID          1082247 non-null  object        
 2   ACCT_NAME        1082247 non-null  object        
 3   CIF_ID           1037216 non-null  object        
 4   SOL_ID           1082247 non-null  object        
 5   ACCT_OPN_DATE    1082247 non-null  datetime64[ns]
 6   SCHM_TYPE        1082247 non-null  object        
 7   SCHM_CODE        1082247 non-null  object        
 8   ACCT_CRNCY_CODE  1082247 non-null  object        
 9   TRAN_DATE        1082247 non-null  datetime64[ns]
 10  TRAN_ID          1082247 non-null  object        
 11  TRAN_AMT         1082247 non-null  float64       
 12  TRAN_PARTICULAR  1078877 non-null  object        
 13  TRAN_RMKS        887652 non-null   object        
 14  PART_TR

In [21]:
txns[['ACCT_NAME','SCHM_TYPE','ACCT_CRNCY_CODE']].head()

,ACCT_NAME,SCHM_TYPE,ACCT_CRNCY_CODE
0,DOREEN MUSIIMENTA,ODA,USD
1,DOREEN MUSIIMENTA,ODA,USD
2,DOREEN MUSIIMENTA,ODA,USD
3,DOREEN MUSIIMENTA,ODA,USD
4,DOREEN MUSIIMENTA,ODA,USD


In [22]:
account_activity = (
    txns.groupby('ACID')
    .size()
    .sort_values()
)

account_activity.describe()

count    20512.000000
mean        52.761652
std        303.128575
min          1.000000
25%          3.000000
50%         12.000000
75%         44.000000
max      36890.000000
dtype: float64

In [23]:
txns[
    (txns['SCHM_TYPE']=='ODA') &
    (txns['ACCT_CRNCY_CODE']=='USD')
]['ACID'].nunique()

17069

In [24]:
txns.groupby('ACID')['ACCT_OPN_DATE'].first().describe()

count                            20512
mean     2025-04-06 03:31:01.778470912
min                2024-01-02 00:00:00
25%                2024-08-08 00:00:00
50%                2025-03-26 00:00:00
75%                2025-11-27 00:00:00
max                2026-07-30 00:00:00
Name: ACCT_OPN_DATE, dtype: object

In [25]:
def account_profile(df):
    credits = df[df['PART_TRAN_TYPE']=='C']
    debits = df[df['PART_TRAN_TYPE']=='D']

    return pd.Series({
        'transaction_count': len(df),

        'credit_count': len(credits),
        'debit_count': len(debits),

        'total_credit': credits['TRAN_AMT'].sum(),
        'total_debit': debits['TRAN_AMT'].sum(),

        'avg_credit': credits['TRAN_AMT'].mean(),
        'avg_debit': debits['TRAN_AMT'].mean(),

        'max_credit': credits['TRAN_AMT'].max(),
        'max_debit': debits['TRAN_AMT'].max(),

        'unique_credit_parties':
            credits['COUNTERPARTY'].nunique(),

        'unique_debit_parties':
            debits['COUNTERPARTY'].nunique()
    })


profiles = (
    txns
    .groupby('ACID')
    .apply(account_profile)
    .reset_index()
)

profiles.head()

/tmp/ipykernel_1907844/853639827.py:31: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(account_profile)


,ACID,transaction_count,credit_count,debit_count,total_credit,total_debit,avg_credit,avg_debit,max_credit,max_debit,unique_credit_parties,unique_debit_parties
0,UG5922760,79.0,24.0,55.0,559410.76,556440.71,23308.781667,10117.103818,59774.30,53730.00,14.0,12.0
1,UG5922821,45.0,12.0,33.0,851.70,851.19,70.975000,25.793636,210.69,72.18,2.0,1.0
2,UG5922880,12.0,3.0,9.0,1350.00,1332.81,450.000000,148.090000,700.00,508.20,2.0,7.0
3,UG5922890,85.0,35.0,50.0,55882.45,55800.00,1596.641429,1116.000000,5000.00,7000.00,11.0,10.0
4,UG5922894,7.0,4.0,3.0,1741.95,1700.00,435.487500,566.666667,1213.00,600.00,4.0,1.0


In [26]:
profiles[
    profiles['ACID']=='UG6246002'
]

,ACID,transaction_count,credit_count,debit_count,total_credit,total_debit,avg_credit,avg_debit,max_credit,max_debit,unique_credit_parties,unique_debit_parties
14998,UG6246002,20.0,11.0,9.0,1331151.0,1328037.29,121013.727273,147559.698889,500000.0,670000.0,11.0,9.0


In [27]:
txns[
    txns['ACID']=='UG6246002'
]

,ACID,FORACID,ACCT_NAME,CIF_ID,SOL_ID,ACCT_OPN_DATE,SCHM_TYPE,SCHM_CODE,ACCT_CRNCY_CODE,TRAN_DATE,TRAN_ID,TRAN_AMT,TRAN_PARTICULAR,TRAN_RMKS,PART_TRAN_TYPE,TRAN_TYPE,TRAN_SUB_TYPE,RCRE_TIME,RPT_CODE,COUNTERPARTY
1012267,UG6246002,9030027282317,FRONTIER TEXTILE DEALERS CO LIMITED,20301186321,031075,2025-11-04,ODA,OD005,USD,2025-11-04,0378139,100.00,ASHIRAF,TEXTILE BUSINESS,C,C,RI,2025-11-04 16:47:10,CMI,ASHIRAF
1099961,UG6246002,9030027282317,FRONTIER TEXTILE DEALERS CO LIMITED,20301186321,031075,2025-11-04,ODA,OD005,USD,2025-11-04,S99264131,88.50,Ac xfr from gl 0603 to 3500,TACBSH,D,T,BI,2025-11-05 02:04:49,TRF,AC XFR FROM GL 0603 TO 3500
1099962,UG6246002,9030027282317,FRONTIER TEXTILE DEALERS CO LIMITED,20301186321,031075,2025-11-04,ODA,OD005,USD,2025-11-04,S99264131,88.50,Ac xfr from gl 0603 to 3500,TACBSH,C,T,BI,2025-11-05 02:04:49,TRF,AC XFR FROM GL 0603 TO 3500
1419621,UG6246002,9030027282317,FRONTIER TEXTILE DEALERS CO LIMITED,20301186321,031075,2025-11-04,ODA,OD005,USD,2026-03-31,S37340109,99900.15,payment for,OT26090UG0109567,D,T,CI,2026-03-31 09:31:27,TRF,PAYMENT FOR
1426283,UG6246002,9030027282317,FRONTIER TEXTILE DEALERS CO LIMITED,20301186321,031075,2025-11-04,ODA,OD005,USD,2026-03-31,S37632727,1.61,9030027282317:Int.Coll:01-03-2026 to 31-03-2026,Interest run,D,T,IC,2026-04-01 02:31:52,INT,9030027282317:INT.COLL:01-03-2026 TO 31-03-2026
1426318,UG6246002,9030027282317,FRONTIER TEXTILE DEALERS CO LIMITED,20301186321,031075,2025-11-04,ODA,OD005,USD,2026-03-30,S37125360,99985.00,TERENCE A WALTS,IT26089UG0175304,C,T,CI,2026-03-30 19:23:28,TRF,TERENCE A WALTS
1429099,UG6246002,9030027282317,FRONTIER TEXTILE DEALERS CO LIMITED,20301186321,031075,2025-11-04,ODA,OD005,USD,2026-04-01,S37768270,17000.00,MIE SUGIYAMA,IT26091UG0176129,C,T,CI,2026-04-01 11:02:43,TRF,MIE SUGIYAMA
1434090,UG6246002,9030027282317,FRONTIER TEXTILE DEALERS CO LIMITED,20301186321,031075,2025-11-04,ODA,OD005,USD,2026-04-01,S37707558,350000.00,1/WEI FU,IT26091UG0175981,C,T,CI,2026-04-01 08:01:20,TRF,WEI FU
1436926,UG6246002,9030027282317,FRONTIER TEXTILE DEALERS CO LIMITED,20301186321,031075,2025-11-04,ODA,OD005,USD,2026-04-07,S39188230,230307.50,1/PERAZZINI PIERGIUSEPPE,IT26097UG0177536,C,T,CI,2026-04-07 08:04:29,TRF,PERAZZINI PIERGIUSEPPE
1436927,UG6246002,9030027282317,FRONTIER TEXTILE DEALERS CO LIMITED,20301186321,031075,2025-11-04,ODA,OD005,USD,2026-04-07,S39188586,84997.00,CHARLES RAHILL LLC,IT26097UG0177614,C,T,CI,2026-04-07 08:04:59,TRF,CHARLES RAHILL LLC


### Flow through ratio

In [28]:
profiles['flow_through_ratio'] = (
    profiles['total_debit'] /
    profiles['total_credit']
)

In [29]:
profiles[
    profiles['ACID']=='UG6246002'
]

,ACID,transaction_count,credit_count,debit_count,total_credit,total_debit,avg_credit,avg_debit,max_credit,max_debit,unique_credit_parties,unique_debit_parties,flow_through_ratio
14998,UG6246002,20.0,11.0,9.0,1331151.0,1328037.29,121013.727273,147559.698889,500000.0,670000.0,11.0,9.0,0.997661


Almost every dollar that enters leaves

### Largest credit concentration

In [31]:
# largest_credit_share = (
#     max_credit /
#     total_credit
# )

In [32]:
import numpy as np

profiles['flow_through_ratio'] = (
    profiles['total_debit'] / profiles['total_credit']
).replace([np.inf, -np.inf], np.nan)

profiles['net_retained'] = (
    profiles['total_credit'] - profiles['total_debit']
)

profiles['retention_ratio'] = (
    profiles['net_retained'] / profiles['total_credit']
).replace([np.inf, -np.inf], np.nan)

profiles['credit_debit_ratio'] = (
    profiles['credit_count'] / profiles['debit_count']
).replace([np.inf, -np.inf], np.nan)

profiles['largest_credit_share'] = (
    profiles['max_credit'] / profiles['total_credit']
).replace([np.inf, -np.inf], np.nan)

profiles['largest_debit_share'] = (
    profiles['max_debit'] / profiles['total_debit']
).replace([np.inf, -np.inf], np.nan)

In [33]:
profiles.loc[
    profiles['ACID']=='UG6246002',
    [
        'flow_through_ratio',
        'retention_ratio',
        'largest_credit_share',
        'largest_debit_share'
    ]
]

,flow_through_ratio,retention_ratio,largest_credit_share,largest_debit_share
14998,0.997661,0.002339,0.375615,0.504504


### Account Age

In [34]:
snapshot_date = txns['TRAN_DATE'].max()
snapshot_date

Timestamp('2026-07-30 00:00:00')

In [35]:
acct_age = (
    txns.groupby('ACID')['ACCT_OPN_DATE']
    .first()
)

profiles['account_age_days'] = (
    snapshot_date - acct_age
).dt.days.values

In [36]:
profiles.loc[
    profiles['ACID']=='UG6246002',
    ['account_age_days']
]

,account_age_days
14998,268


### Meta data

In [37]:
account_info = (
    txns.groupby('ACID')
       .agg({
           'FORACID':'first',
           'ACCT_NAME':'first',
           'SOL_ID':'first',
           'SCHM_TYPE':'first',
           'SCHM_CODE':'first',
           'ACCT_CRNCY_CODE':'first',
           'ACCT_OPN_DATE':'first'
       })
       .reset_index()
)

profiles = profiles.merge(account_info, on='ACID')

In [38]:
profiles['turnover'] = (
    profiles['total_credit'] + profiles['total_debit']
)

profiles[['turnover']].describe()

,turnover
count,2.051200e+04
mean,9.608570e+05
std,3.592874e+07
min,2.700000e-01
25%,5.893075e+02
50%,1.322291e+04
75%,6.504728e+04
max,3.888169e+09


In [39]:
profiles.nlargest(
    20,
    'turnover'
)[[
    'ACCT_NAME',
    'turnover',
    'transaction_count',
    'flow_through_ratio',
    'retention_ratio'
]]

,ACCT_NAME,turnover,transaction_count,flow_through_ratio,retention_ratio
3726,UGANDA NATIONAL OIL COMPANY RBA,3.888169e+09,7863.0,0.995946,4.053798e-03
937,BANK OF NEWYORK MELLON,2.510837e+09,36890.0,1.015852,-1.585151e-02
16720,UGANDA NATIONAL OIL COMPANY LTD BORROWER RBA,1.941529e+09,105.0,0.832888,1.671121e-01
6073,JUBA EXPRESS FOREX BUREEAU &MONEY TRANSFER LIM...,5.856942e+08,1690.0,0.999043,9.571161e-04
3138,MALYAM GENERAL TRADING CO. LTD,4.348853e+08,3169.0,0.997562,2.438010e-03
10396,UMEME ADMITTED BUY-OUT,3.399640e+08,38.0,1.847303,-8.473035e-01
6383,TOTALENERGIES MARKETING UGANDA LIMITED,3.277126e+08,104.0,1.000000,1.818809e-16
12496,UGANDA NATIONAL OIL COMPANY LIMITED,3.207537e+08,23.0,0.996266,3.734204e-03
7301,ENSIBUUKO TECH LIMITED,2.443509e+08,71.0,0.999977,2.296417e-05
4030,PRIMACY SOURCING (U) LIMITED,2.280160e+08,2510.0,0.999585,4.151476e-04


In [40]:
profiles.loc[
    profiles['ACID']=='UG6246002'
].T

,14998
ACID,UG6246002
transaction_count,20.0
credit_count,11.0
debit_count,9.0
total_credit,1331151.0
total_debit,1328037.29
avg_credit,121013.727273
avg_debit,147559.698889
max_credit,500000.0
max_debit,670000.0


### Velocity

In [41]:
def velocity_features(df):
    df = df.sort_values('RCRE_TIME')

    credits = df[df['PART_TRAN_TYPE'] == 'C'][['RCRE_TIME', 'TRAN_AMT']].copy()
    debits = df[df['PART_TRAN_TYPE'] == 'D'][['RCRE_TIME', 'TRAN_AMT']].copy()

    if credits.empty or debits.empty:
        return pd.Series({
            'avg_hours_to_next_debit': np.nan,
            'median_hours_to_next_debit': np.nan,
            'credits_followed_24h': 0,
            'credits_followed_48h': 0
        })

    debits['DEBIT_TIME'] = debits['RCRE_TIME']

    merged = pd.merge_asof(
        credits.sort_values('RCRE_TIME'),
        debits.sort_values('RCRE_TIME'),
        on='RCRE_TIME',
        direction='forward',
        suffixes=('_credit', '_debit')
    )

    merged['hours'] = (
        merged['DEBIT_TIME'] - merged['RCRE_TIME']
    ).dt.total_seconds() / 3600

    return pd.Series({
        'avg_hours_to_next_debit': merged['hours'].mean(),
        'median_hours_to_next_debit': merged['hours'].median(),
        'credits_followed_24h': (merged['hours'] <= 24).sum(),
        'credits_followed_48h': (merged['hours'] <= 48).sum()
    })

In [42]:
velocity_profiles = (
    txns
    .groupby('ACID')
    .apply(velocity_features)
    .reset_index()
)

profiles = profiles.merge(velocity_profiles, on='ACID', how='left')

/tmp/ipykernel_1907844/321051853.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(velocity_features)


In [44]:
profiles.loc[
    profiles['ACID'] == 'UG6246002',
    [
        'avg_hours_to_next_debit',
        'median_hours_to_next_debit',
        'credits_followed_24h',
        'credits_followed_48h'
    ]
]

,avg_hours_to_next_debit,median_hours_to_next_debit,credits_followed_24h,credits_followed_48h
14998,584.392528,58.510139,4.0,5.0


In [45]:
profiles['risk_score'] = 0

profiles.loc[profiles['flow_through_ratio'] > 0.95, 'risk_score'] += 2
profiles.loc[profiles['largest_credit_share'] > 0.30, 'risk_score'] += 2
profiles.loc[profiles['credits_followed_24h'] >= 3, 'risk_score'] += 2
profiles.loc[profiles['account_age_days'] < 365, 'risk_score'] += 1
profiles.loc[profiles['turnover'] > 1_000_000, 'risk_score'] += 1

In [46]:
def concentration_features(df):
    credits = (
        df[df['PART_TRAN_TYPE'] == 'C']
        .sort_values('TRAN_AMT', ascending=False)
    )

    debits = (
        df[df['PART_TRAN_TYPE'] == 'D']
        .sort_values('TRAN_AMT', ascending=False)
    )

    def top_share(trans):
        if trans.empty:
            return pd.Series({
                'top3_credit_share': np.nan,
                'gini_proxy_credit': np.nan
            })

        total = trans['TRAN_AMT'].sum()

        top3 = trans['TRAN_AMT'].head(3).sum() / total

        return pd.Series({
            'top3_credit_share': top3,
            'gini_proxy_credit': trans['TRAN_AMT'].std() / trans['TRAN_AMT'].mean()
        })

    c = top_share(credits)

    if debits.empty:
        d = pd.Series({
            'top3_debit_share': np.nan,
            'gini_proxy_debit': np.nan
        })
    else:
        total = debits['TRAN_AMT'].sum()

        d = pd.Series({
            'top3_debit_share':
                debits['TRAN_AMT'].head(3).sum() / total,

            'gini_proxy_debit':
                debits['TRAN_AMT'].std() / debits['TRAN_AMT'].mean()
        })

    return pd.concat([c, d])

In [47]:
conc = (
    txns
    .groupby('ACID')
    .apply(concentration_features)
    .reset_index()
)

profiles = profiles.merge(conc, on='ACID')

/tmp/ipykernel_1907844/2179620720.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(concentration_features)


## Similarity search

In [48]:
feature_cols = [
    'transaction_count',
    'credit_count',
    'debit_count',
    'total_credit',
    'total_debit',
    'avg_credit',
    'avg_debit',
    'max_credit',
    'max_debit',
    'flow_through_ratio',
    'retention_ratio',
    'largest_credit_share',
    'largest_debit_share',
    'account_age_days',
    'avg_hours_to_next_debit',
    'median_hours_to_next_debit',
    'credits_followed_24h',
    'credits_followed_48h',
    'top3_credit_share',
    'top3_debit_share'
]

In [49]:
from sklearn.preprocessing import StandardScaler

X = profiles[feature_cols].fillna(0)

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [50]:
from sklearn.neighbors import NearestNeighbors

nn = NearestNeighbors(
    n_neighbors=11,
    metric='euclidean'
)

nn.fit(X_scaled)

NearestNeighbors(metric='euclidean', n_neighbors=11)

In [51]:
frontier_idx = profiles.index[
    profiles['ACID'] == 'UG6246002'
][0]

In [52]:
distances, indices = nn.kneighbors(
    X_scaled[frontier_idx].reshape(1, -1)
)

similar_accounts = (
    profiles
    .iloc[indices[0]]
    .copy()
)

similar_accounts['distance'] = distances[0]

similar_accounts[
    [
        'ACCT_NAME',
        'FORACID',
        'turnover',
        'flow_through_ratio',
        'account_age_days',
        'distance'
    ]
]

,ACCT_NAME,FORACID,turnover,flow_through_ratio,account_age_days,distance
14998,FRONTIER TEXTILE DEALERS CO LIMITED,9030027282317,2659188.29,0.997661,268,0.000000
14094,EVERSTONE INDUSTRIES (U) LTD,9030027094707,1178289.86,0.902063,308,0.726900
15304,HIMANA TRANSPORTATION-SMC LIMITED,9030027373797,2254064.65,0.999213,251,0.777770
15068,JUA MILELE INVESTMENT AND TRADING CO. LTD,9030027300064,720948.16,0.951137,265,0.778299
15898,KIM REAL ESTATES CO LTD,9030027499499,2846803.83,0.999597,219,0.789870
15806,GLOBAL LEAF HOLDINGS (U) LIMITED,9030027483428,1402534.00,1.144849,224,0.794089
16751,CABLINKS ESTABLISHMENTS LIMITED,9030027735672,1967214.86,0.999334,167,0.800835
13830,YUN AN MINING CO (U) LTD,9030027030571,1104877.00,0.995124,318,0.806258
15278,CHEMIROX INDUSTRIAL CHEMICALS LIMITED,9030027365328,150684.00,0.985414,252,0.808265
13491,KIBIRIGE MOHAMED,9030026942135,551629.06,0.781172,335,0.827311


In [56]:
# --- 1a. Which features drive the neighbour selection? ---
ref_vec        = X_scaled[frontier_idx]
neighbour_vecs = X_scaled[indices[0]]          # the 11 rows NN returned (incl. Frontier)

# squared component-wise gap, summed across the neighbours
contrib = ((neighbour_vecs - ref_vec) ** 2).sum(axis=0)

contrib_df = pd.DataFrame({
    'distance_contribution': contrib,
    'pct_of_total': (contrib / contrib.sum() * 100).round(1)
}, index=feature_cols).sort_values('distance_contribution', ascending=False)

print(contrib_df)

                            distance_contribution  pct_of_total
largest_debit_share                      1.180510          18.7
avg_hours_to_next_debit                  1.033435          16.4
avg_debit                                0.626514           9.9
top3_credit_share                        0.585753           9.3
max_debit                                0.539943           8.5
top3_debit_share                         0.521970           8.3
max_credit                               0.510362           8.1
avg_credit                               0.361507           5.7
largest_credit_share                     0.340788           5.4
account_age_days                         0.325137           5.1
credits_followed_24h                     0.058186           0.9
debit_count                              0.047640           0.8
median_hours_to_next_debit               0.046265           0.7
transaction_count                        0.039659           0.6
credits_followed_48h                    

In [57]:
# --- 1b. How correlated are the size features? ---
size_cols = ['transaction_count', 'credit_count', 'debit_count',
             'total_credit', 'total_debit',
             'avg_credit', 'avg_debit', 'max_credit', 'max_debit']

print(profiles[size_cols].corr().round(2))

                   transaction_count  credit_count  debit_count  total_credit  \
transaction_count               1.00          0.42         0.97          0.57   
credit_count                    0.42          1.00         0.19          0.64   
debit_count                     0.97          0.19         1.00          0.45   
total_credit                    0.57          0.64         0.45          1.00   
total_debit                     0.59          0.65         0.47          1.00   
avg_credit                      0.03          0.01         0.03          0.25   
avg_debit                       0.03          0.16        -0.00          0.35   
max_credit                      0.12          0.06         0.11          0.32   
max_debit                       0.11          0.16         0.07          0.39   

                   total_debit  avg_credit  avg_debit  max_credit  max_debit  
transaction_count         0.59        0.03       0.03        0.12       0.11  
credit_count              0.65 

In [58]:
# (a) confirm retention is just the mirror of flow-through
print((profiles['retention_ratio'] + profiles['flow_through_ratio']).describe())
#     -> mean/min/max should all sit at ~1.0

# (b) how outlier-heavy are the ratio + velocity features?
check = ['flow_through_ratio', 'largest_credit_share', 'largest_debit_share',
         'top3_credit_share', 'top3_debit_share',
         'median_hours_to_next_debit', 'avg_hours_to_next_debit']
print(profiles[check].describe(percentiles=[.5, .9, .99]).round(2))

count    2.012100e+04
mean     1.000000e+00
std      6.541842e-15
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      1.000000e+00
dtype: float64
       flow_through_ratio  largest_credit_share  largest_debit_share  \
count            20121.00              20121.00             18787.00   
mean                 4.22                  0.53                 0.53   
std                248.81                  0.32                 0.36   
min                  0.00                  0.00                 0.00   
50%                  0.96                  0.50                 0.48   
90%                  1.00                  1.00                 1.00   
99%                 11.91                  1.00                 1.00   
max              33334.00                  1.00                 1.00   

       top3_credit_share  top3_debit_share  median_hours_to_next_debit  \
count           20121.00          18787.00                    17091.00   
mean        

In [59]:
import numpy as np
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import NearestNeighbors

profiles = profiles.reset_index(drop=True)          # keep index positional == array rows

# --- derived features ---
# normalized velocity: fraction of credits followed by a debit within 24h (size-independent)
profiles['frac_credits_followed_24h'] = (
    profiles['credits_followed_24h'] / profiles['credit_count']
).replace([np.inf, -np.inf], np.nan)

# log intensity: mild size/maturity prior instead of raw volume columns
profiles['log_turnover_per_day'] = np.log1p(
    profiles['turnover_per_day'].replace([np.inf, -np.inf], np.nan)
)

# --- curated behavioural feature set ---
behaviour_cols = [
    'flow_through_ratio',
    'largest_credit_share',
    'largest_debit_share',
    'top3_credit_share',
    'top3_debit_share',
    'median_hours_to_next_debit',
    'frac_credits_followed_24h',
    'account_age_days',
    'log_turnover_per_day',
]

# interpretable caps for the two fat-tailed features
clips = {
    'flow_through_ratio': (0, 2.0),          # outflows >=2x inflows -> "equally high flow-through"
    'median_hours_to_next_debit': (0, 720),  # cap slowness at 30 days
}

Xb = profiles[behaviour_cols].copy()
for col, (lo, hi) in clips.items():
    Xb[col] = Xb[col].clip(lo, hi)
Xb = Xb.fillna(Xb.median())                  # undefined ratios / no-debit accounts

scaler_b   = RobustScaler()
Xb_scaled  = scaler_b.fit_transform(Xb)

nn_b = NearestNeighbors(n_neighbors=11, metric='euclidean').fit(Xb_scaled)

frontier_idx = profiles.index[profiles['ACID'] == 'UG6246002'][0]
dist_b, idx_b = nn_b.kneighbors(Xb_scaled[frontier_idx].reshape(1, -1))

# --- diagnostic 1: do the behavioural features now actually drive distance? ---
ref_b   = Xb_scaled[frontier_idx]
neigh_b = Xb_scaled[idx_b[0]]
contrib_b = ((neigh_b - ref_b) ** 2).sum(axis=0)
print(pd.DataFrame({
    'distance_contribution': contrib_b,
    'pct_of_total': (contrib_b / contrib_b.sum() * 100).round(1)
}, index=behaviour_cols).sort_values('distance_contribution', ascending=False), "\n")

# --- diagnostic 2: old vs new neighbour set ---
def frame(idx_arr, dist_arr):
    f = profiles.iloc[idx_arr][['ACCT_NAME', 'flow_through_ratio',
                                'account_age_days', 'turnover']].copy()
    f['distance'] = dist_arr.round(3)
    return f.reset_index(drop=True)

old_set = set(profiles.iloc[indices[0]]['ACID'])     # from your original StandardScaler run
new_set = set(profiles.iloc[idx_b[0]]['ACID'])

print("OLD neighbours (StandardScaler, full feature set):")
print(frame(indices[0], distances[0]).to_string(index=False), "\n")
print("NEW neighbours (RobustScaler, curated + clipped):")
print(frame(idx_b[0], dist_b[0]).to_string(index=False), "\n")

print("Dropped out :", old_set - new_set)
print("Newly in    :", new_set - old_set)
print("Retained    :", len(old_set & new_set), "of 11")

                            distance_contribution  pct_of_total
log_turnover_per_day                     0.901669          26.8
top3_credit_share                        0.632596          18.8
frac_credits_followed_24h                0.437801          13.0
median_hours_to_next_debit               0.427648          12.7
largest_debit_share                      0.354152          10.5
account_age_days                         0.192651           5.7
largest_credit_share                     0.178119           5.3
top3_debit_share                         0.122840           3.7
flow_through_ratio                       0.111956           3.3 

OLD neighbours (StandardScaler, full feature set):
                                ACCT_NAME  flow_through_ratio  account_age_days   turnover  distance
      FRONTIER TEXTILE DEALERS CO LIMITED            0.997661               268 2659188.29     0.000
             EVERSTONE INDUSTRIES (U) LTD            0.902063               308 1178289.86     0.727
    

In [60]:
fingerprint_cols = [
    'flow_through_ratio', 'largest_credit_share', 'largest_debit_share',
    'top3_credit_share', 'top3_debit_share',
    'median_hours_to_next_debit', 'frac_credits_followed_24h',
    'account_age_days', 'turnover_per_day',
]

old_ids = [i for i in indices[0] if i != frontier_idx]
new_ids = [i for i in idx_b[0]   if i != frontier_idx]

fp = pd.DataFrame({
    'FRONTIER':   profiles.loc[frontier_idx, fingerprint_cols],
    'OLD_median': profiles.iloc[old_ids][fingerprint_cols].median(),
    'NEW_median': profiles.iloc[new_ids][fingerprint_cols].median(),
}).round(2)
print(fp, "\n")

# behaviour-only closeness (size prior excluded), in the robust-scaled space
behaviour_only = [c for c in behaviour_cols if c != 'log_turnover_per_day']
cols_idx = [behaviour_cols.index(c) for c in behaviour_only]
ref_bo   = Xb_scaled[frontier_idx][cols_idx]

def mean_abs_dev(id_list):
    return np.abs(Xb_scaled[id_list][:, cols_idx] - ref_bo).mean()

print("Mean behavioural deviation from Frontier (lower = tighter, size excluded):")
print("  OLD set:", round(mean_abs_dev(old_ids), 3))
print("  NEW set:", round(mean_abs_dev(new_ids), 3))

                               FRONTIER  OLD_median  NEW_median
flow_through_ratio             0.997661        0.99        0.99
largest_credit_share           0.375615        0.35        0.33
largest_debit_share            0.504504        0.40        0.41
top3_credit_share              0.811559        0.76        0.76
top3_debit_share               0.726869        0.71        0.72
median_hours_to_next_debit    58.510139       53.24       53.61
frac_credits_followed_24h      0.363636        0.33        0.33
account_age_days                    268      258.50      255.50
turnover_per_day            9922.344366     5043.46     3093.05 

Mean behavioural deviation from Frontier (lower = tighter, size excluded):
  OLD set: 0.192
  NEW set: 0.136


In [61]:
# ============================================================
#  FINAL REASON LAYER — keyed to the locked feature set
# ============================================================
import numpy as np
import pandas as pd

REF_ACID = 'UG6246002'
ref = profiles.loc[profiles['ACID'] == REF_ACID].iloc[0]

# data-driven "notable" thresholds from the screened population
P = {
    'largest_credit_p75': profiles['largest_credit_share'].quantile(0.75),
    'largest_debit_p75' : profiles['largest_debit_share'].quantile(0.75),
    'turnover_day_p90'  : profiles['turnover_per_day'].quantile(0.90),
}

def _pct(x): return "n/a" if pd.isna(x) else f"{x*100:.0f}%"
def _num(x): return "n/a" if pd.isna(x) else f"{x:,.0f}"

def generate_reason(row):
    hits = []

    if pd.notna(row['account_age_days']) and row['account_age_days'] <= 365:
        hits.append(f"recently opened ({int(row['account_age_days'])}d; ref {int(ref['account_age_days'])}d)")

    if pd.notna(row['flow_through_ratio']) and row['flow_through_ratio'] >= 0.90:
        hits.append(f"high fund flow-through, limited retention ({_pct(row['flow_through_ratio'])} of inflows moved out)")

    if pd.notna(row['median_hours_to_next_debit']) and row['median_hours_to_next_debit'] <= 72:
        hits.append(f"rapid onward movement (~{row['median_hours_to_next_debit']:.0f}h median to next debit)")

    if pd.notna(row['frac_credits_followed_24h']) and row['frac_credits_followed_24h'] >= 0.30:
        hits.append(f"inflows commonly moved within 24h ({_pct(row['frac_credits_followed_24h'])} of credits)")

    if pd.notna(row['largest_credit_share']) and row['largest_credit_share'] >= P['largest_credit_p75']:
        hits.append(f"incoming value concentrated ({_pct(row['largest_credit_share'])} in largest credit)")

    if pd.notna(row['largest_debit_share']) and row['largest_debit_share'] >= P['largest_debit_p75']:
        hits.append(f"outgoing value concentrated ({_pct(row['largest_debit_share'])} in largest debit)")

    if pd.notna(row['turnover_per_day']) and row['turnover_per_day'] >= P['turnover_day_p90']:
        hits.append(f"high daily turnover vs peers ({_num(row['turnover_per_day'])}/day)")

    if not hits:
        return "Behavioural profile close to reference account (no single indicator dominant)"

    return "Similar transaction behaviour to reference account: " + "; ".join(hits)


# --- apply to the NEW neighbour set (built from profiles, all cols present) ---
pairs    = [(i, d) for i, d in zip(idx_b[0], dist_b[0]) if i != frontier_idx]
new_ids  = [i for i, d in pairs]
new_dist = [d for i, d in pairs]

review = profiles.iloc[new_ids].copy()
review['distance']    = new_dist
review['risk_reason'] = review.apply(generate_reason, axis=1)
review = review.sort_values('distance').reset_index(drop=True)

# --- read the top 10 ---
for _, r in review.iterrows():
    print(f"\n{r['ACCT_NAME']}  |  dist {r['distance']:.3f}  |  {r['FORACID']}")
    print(f"   {r['risk_reason']}")

# --- differentiation check (the original failure mode) ---
print("\n\nUnique reason strings:", review['risk_reason'].nunique(), "of", len(review))
print(review['risk_reason'].value_counts())


JUA MILELE INVESTMENT AND TRADING CO. LTD  |  dist 0.411  |  9030027300064
   Similar transaction behaviour to reference account: recently opened (265d; ref 268d); high fund flow-through, limited retention (95% of inflows moved out); rapid onward movement (~56h median to next debit); inflows commonly moved within 24h (33% of credits); high daily turnover vs peers (2,721/day)

JADEPANDA REAL ESTATE LIMITED  |  dist 0.502  |  9030026813823
   Similar transaction behaviour to reference account: recently opened (356d; ref 268d); high fund flow-through, limited retention (100% of inflows moved out); inflows commonly moved within 24h (40% of credits); high daily turnover vs peers (16,316/day)

GLOBAL LEAF HOLDINGS (U) LIMITED  |  dist 0.506  |  9030027483428
   Similar transaction behaviour to reference account: recently opened (224d; ref 268d); high fund flow-through, limited retention (114% of inflows moved out); rapid onward movement (~46h median to next debit); inflows commonly moved wi

### Patterns

In [53]:
profiles['turnover_per_day'] = (
    profiles['turnover'] /
    profiles['account_age_days']
)

In [54]:
profiles['turnover_per_day'].describe(percentiles=[.5,.75,.9,.95,.99])

count    2.051200e+04
mean              inf
std               NaN
min      5.283757e-04
50%      3.426640e+01
75%      1.678951e+02
90%      8.996419e+02
95%      2.712872e+03
99%      2.865121e+04
max               inf
Name: turnover_per_day, dtype: float64

In [55]:
profiles['avg_transaction'] = (
    profiles['turnover'] /
    profiles['transaction_count']
)

In [59]:
profiles['credit_party_ratio'] = (
    profiles['unique_credit_parties'] /
    profiles['credit_count']
)

profiles['debit_party_ratio'] = (
    profiles['unique_debit_parties'] /
    profiles['debit_count']
)

In [60]:
profiles['largest_transaction_share'] = (
    profiles[
        ['largest_credit_share',
         'largest_debit_share']
    ].max(axis=1)
)

In [61]:
profiles['velocity_score'] = (
    profiles['credits_followed_24h']*2 +
    profiles['credits_followed_48h']
)

In [62]:
txns['round_amount'] = (
    (txns['TRAN_AMT'] % 1000) == 0
)

round_ratio = (
    txns
    .groupby('ACID')['round_amount']
    .mean()
    .rename('round_amount_ratio')
)

profiles = profiles.merge(
    round_ratio,
    on='ACID'
)

In [82]:
def generate_reason(row):
    reasons = []

    if row['account_age_days'] <= 270:
        reasons.append("Recently opened account")

    if row['flow_through_ratio'] >= 0.95:
        reasons.append("High fund flow-through")

    if row['retention_ratio'] <= 0.05:
        reasons.append("Low balance retention")

    if row['median_hours_to_next_debit'] <= 72:
        reasons.append("Rapid movement of funds")

    if row['largest_credit_share'] >= 0.30:
        reasons.append("Large incoming transaction concentration")

    if row['largest_debit_share'] >= 0.30:
        reasons.append("Large outgoing transaction concentration")

    if row['unique_credit_parties'] >= 5:
        reasons.append("Multiple incoming counterparties")

    if row['unique_debit_parties'] >= 5:
        reasons.append("Multiple outgoing counterparties")

    return "; ".join(reasons)

In [83]:
similar_accounts['risk_reason'] = (
    similar_accounts.apply(generate_reason, axis=1)
)

In [ ]:
si

In [77]:
report_columns = [
    'ACCT_NAME',
    'FORACID',
    # 'CIF_ID',
    'SOL_ID',
    'ACCT_OPN_DATE',
    'ACCT_CRNCY_CODE',
    'SCHM_TYPE',

    'transaction_count',
    'turnover',

    'total_credit',
    'total_debit',

    'flow_through_ratio',
    'retention_ratio',

    'median_hours_to_next_debit',

    'unique_credit_parties',
    'unique_debit_parties',

    'largest_credit_share',
    'largest_debit_share',

    # 'similarity_score',

    'risk_score',
    'risk_reason'
]

In [78]:
similar_accounts.columns

Index(['ACID', 'transaction_count', 'credit_count', 'debit_count',
       'total_credit', 'total_debit', 'avg_credit', 'avg_debit', 'max_credit',
       'max_debit', 'unique_credit_parties', 'unique_debit_parties',
       'flow_through_ratio', 'net_retained', 'retention_ratio',
       'credit_debit_ratio', 'largest_credit_share', 'largest_debit_share',
       'account_age_days', 'FORACID', 'ACCT_NAME', 'SOL_ID', 'SCHM_TYPE',
       'SCHM_CODE', 'ACCT_CRNCY_CODE', 'ACCT_OPN_DATE', 'turnover',
       'avg_hours_to_next_debit', 'median_hours_to_next_debit',
       'credits_followed_24h', 'credits_followed_48h', 'risk_score',
       'top3_credit_share', 'gini_proxy_credit', 'top3_debit_share',
       'gini_proxy_debit', 'distance', 'risk_reason'],
      dtype='object')

In [86]:
similar_accounts[report_columns].to_csv("similar.csv", index=False)

In [84]:
similar_accounts[['ACCT_NAME','distance','risk_reason']]

,ACCT_NAME,distance,risk_reason
15014,FRONTIER TEXTILE DEALERS CO LIMITED,0.000000,Recently opened account; High fund flow-throug...
15823,GLOBAL LEAF HOLDINGS (U) LIMITED,0.684635,Recently opened account; High fund flow-throug...
16773,CABLINKS ESTABLISHMENTS LIMITED,0.734426,Recently opened account; Rapid movement of fun...
15320,HIMANA TRANSPORTATION-SMC LIMITED,0.779738,Recently opened account; High fund flow-throug...
15084,JUA MILELE INVESTMENT AND TRADING CO. LTD,0.781324,Recently opened account; High fund flow-throug...
15915,KIM REAL ESTATES CO LTD,0.792539,Recently opened account; High fund flow-throug...
13835,YUN AN MINING CO (U) LTD,0.810109,High fund flow-through; Low balance retention;...
15294,CHEMIROX INDUSTRIAL CHEMICALS LIMITED,0.810294,Recently opened account; High fund flow-throug...
13496,KIBIRIGE MOHAMED,0.818890,Large incoming transaction concentration; Larg...
14231,MARKH AUTOMOBILE CO. LTD,0.820933,High fund flow-through; Low balance retention;...


In [ ]:
Ravo Tech Company SMC Ltd	9030028241746	UG6370211


In [ ]:
Nebulacore Innovations Limited	9030025978926	UG6119095
Aethergrid Tech Co Ltd	9030025978896	UG6119094
Kwelivest Investment SMC Limited	9030025978918	UG6119032
Pacific Prince International (u) co. Limited	9030025308521	UG6052746

Emic Ventures	9030025905260	UG6112062
Heshima Prime Logistics Limited	9030025725963	UG6094534
Shaupam Enteprises Limited	9030026332531	UG6154756
Omosphire Uganda Limited	9030027884430	UG6310606



In [ ]:
source .env && sqlcmd -S "$AML_WRITE_HOST,$AML_WRITE_PORT" -d "$AML_WRITE_DB" -U "$AML_WRITE_USER" -P "$AML_WRITE_PWD" -Q "INSERT INTO dbo.aml_watchlist (acid, acct_name, notes) VALUES ('UG6370211','RAVO TECH COMPANY SMC LIMITED','Second reference');"

In [ ]:
source .env && sqlcmd -S "$AML_WRITE_HOST,$AML_WRITE_PORT" -d "$AML_WRITE_DB" -U "$AML_WRITE_USER" -P "$AML_WRITE_PWD" -W -Q "SELECT TOP 12 rank_within_ref AS rk, priority_band AS band, CAST(distance AS DECIMAL(5,3)) AS dist, account_status AS status, match_name FROM dbo.aml_screening_results WHERE reference_acid='UG6370211' AND run_date=CAST(GETDATE() AS DATE) ORDER BY distance;"

In [ ]:
source .env && sqlcmd -S "$AML_WRITE_HOST,$AML_WRITE_PORT" -d "$AML_WRITE_DB" -U "$AML_WRITE_USER" -P "$AML_WRITE_PWD" -Q "INSERT INTO dbo.aml_watchlist (acid, acct_name, notes) VALUES ('UG6119095','Nebulacore Innovations Limited','Confirmed mule');"

In [ ]:
source .env && sqlcmd -S "$AML_WRITE_HOST,$AML_WRITE_PORT" -d "$AML_WRITE_DB" -U "$AML_WRITE_USER" -P "$AML_WRITE_PWD" -Q "INSERT INTO dbo.aml_watchlist (acid, acct_name, notes) VALUES ('UG6119094','Aethergrid Tech Co Ltd','Confirmed mule');"

In [ ]:
source .env && sqlcmd -S "$AML_WRITE_HOST,$AML_WRITE_PORT" -d "$AML_WRITE_DB" -U "$AML_WRITE_USER" -P "$AML_WRITE_PWD" -Q "INSERT INTO dbo.aml_watchlist (acid, acct_name, notes) VALUES ('UG6119032','Kwelivest Investment SMC Limited','Confirmed mule');"

In [ ]:
source .env && sqlcmd -S "$AML_WRITE_HOST,$AML_WRITE_PORT" -d "$AML_WRITE_DB" -U "$AML_WRITE_USER" -P "$AML_WRITE_PWD" -Q "INSERT INTO dbo.aml_watchlist (acid, acct_name, notes) VALUES ('UG6052746','Pacific Prince International (u) co. Limited','Confirmed mule');"

In [ ]:
source .env && sqlcmd -S "$AML_WRITE_HOST,$AML_WRITE_PORT" -d "$AML_WRITE_DB" -U "$AML_WRITE_USER" -P "$AML_WRITE_PWD" -Q "INSERT INTO dbo.aml_watchlist (acid, acct_name, notes) VALUES ('UG6112062','Emic Ventures','Confirmed mule');"

In [ ]:
source .env && sqlcmd -S "$AML_WRITE_HOST,$AML_WRITE_PORT" -d "$AML_WRITE_DB" -U "$AML_WRITE_USER" -P "$AML_WRITE_PWD" -Q "INSERT INTO dbo.aml_watchlist (acid, acct_name, notes) VALUES ('UG6094534','Heshima Prime Logistics Limited','Confirmed mule');"

In [ ]:
source .env && sqlcmd -S "$AML_WRITE_HOST,$AML_WRITE_PORT" -d "$AML_WRITE_DB" -U "$AML_WRITE_USER" -P "$AML_WRITE_PWD" -Q "INSERT INTO dbo.aml_watchlist (acid, acct_name, notes) VALUES ('UG6154756','Shaupam Enteprises Limited','Confirmed mule');"

In [ ]:
source .env && sqlcmd -S "$AML_WRITE_HOST,$AML_WRITE_PORT" -d "$AML_WRITE_DB" -U "$AML_WRITE_USER" -P "$AML_WRITE_PWD" -Q "INSERT INTO dbo.aml_watchlist (acid, acct_name, notes) VALUES ('UG6310606','Omosphire Uganda Limited','Confirmed mule');"

In [ ]:
source .env && sqlcmd -S "$AML_WRITE_HOST,$AML_WRITE_PORT" -d "$AML_WRITE_DB" -U "$AML_WRITE_USER" -P "$AML_WRITE_PWD" -Q "INSERT INTO dbo.aml_watchlist (acid, acct_name, notes) VALUES ('UG6119095','Nebulacore Innovations Limited','Confirmed mule');"

source .env && sqlcmd -S "$AML_WRITE_HOST,$AML_WRITE_PORT" -d "$AML_WRITE_DB" -U "$AML_WRITE_USER" -P "$AML_WRITE_PWD" -Q "INSERT INTO dbo.aml_watchlist (acid, acct_name, notes) VALUES ('UG6119094','Aethergrid Tech Co Ltd','Confirmed mule');"

source .env && sqlcmd -S "$AML_WRITE_HOST,$AML_WRITE_PORT" -d "$AML_WRITE_DB" -U "$AML_WRITE_USER" -P "$AML_WRITE_PWD" -Q "INSERT INTO dbo.aml_watchlist (acid, acct_name, notes) VALUES ('UG6119032','Kwelivest Investment SMC Limited','Confirmed mule');"

source .env && sqlcmd -S "$AML_WRITE_HOST,$AML_WRITE_PORT" -d "$AML_WRITE_DB" -U "$AML_WRITE_USER" -P "$AML_WRITE_PWD" -Q "INSERT INTO dbo.aml_watchlist (acid, acct_name, notes) VALUES ('UG6052746','Pacific Prince International (u) co. Limited','Confirmed mule');"

source .env && sqlcmd -S "$AML_WRITE_HOST,$AML_WRITE_PORT" -d "$AML_WRITE_DB" -U "$AML_WRITE_USER" -P "$AML_WRITE_PWD" -Q "INSERT INTO dbo.aml_watchlist (acid, acct_name, notes) VALUES ('UG6112062','Emic Ventures','Confirmed mule');"

source .env && sqlcmd -S "$AML_WRITE_HOST,$AML_WRITE_PORT" -d "$AML_WRITE_DB" -U "$AML_WRITE_USER" -P "$AML_WRITE_PWD" -Q "INSERT INTO dbo.aml_watchlist (acid, acct_name, notes) VALUES ('UG6094534','Heshima Prime Logistics Limited','Confirmed mule');"

source .env && sqlcmd -S "$AML_WRITE_HOST,$AML_WRITE_PORT" -d "$AML_WRITE_DB" -U "$AML_WRITE_USER" -P "$AML_WRITE_PWD" -Q "INSERT INTO dbo.aml_watchlist (acid, acct_name, notes) VALUES ('UG6154756','Shaupam Enteprises Limited','Confirmed mule');"

source .env && sqlcmd -S "$AML_WRITE_HOST,$AML_WRITE_PORT" -d "$AML_WRITE_DB" -U "$AML_WRITE_USER" -P "$AML_WRITE_PWD" -Q "INSERT INTO dbo.aml_watchlist (acid, acct_name, notes) VALUES ('UG6310606','Omosphire Uganda Limited','Confirmed mule');"